# 02. PDF를 Markdown으로 변환한 RAG 개선 실습

## 실습 목적

이 실습에서는 PDF 문서를 바로 로딩하지 않고 Markdown 문서로 변환한 뒤 RAG를 구성한다.  
PDF 직접 로딩 방식과 비교하여 문서 구조, 표, 제목 정보가 검색 품질에 어떤 영향을 주는지 확인한다.

## Advanced RAG 개선 방향
```
Naive RAG:    질문 → 검색 → 생성
Advanced RAG: 
- PDF를 Markdown으로 변환하여 문서 구조를 보존한 뒤 로딩
- 질문 → [쿼리 재작성 / 다중쿼리] → 검색 → [재순위화 / 압축] → 생성
```

## 학습 목표

이번 실습에서는 PDF 문서를 Markdown으로 변환한 뒤 RAG를 구성하고, PDF 직접 로딩 방식과 Markdown 기반 로딩 방식의 차이를 확인한다.

실습 후 다음 내용을 설명할 수 있어야 한다.

1. PDF 문서를 Markdown으로 변환하는 이유
2. Markdown 변환 후에도 정제가 필요한 이유
3. Markdown 문서를 `TextLoader`로 로딩하는 방법
4. Markdown 기반 청크 분할 방식의 특징
5. PDF 기반 RAG와 Markdown 기반 RAG의 검색 결과 차이
6. Markdown 변환 방식이 표, 제목, 문단 구조 보존에 미치는 영향

## 실습 데이터
**「2026 AI 주요 트렌드 전망」NIA AI 트렌드 보고서 기반 한국어 RAG 챗봇 만들기** 

In [1]:

from dotenv import load_dotenv
load_dotenv(override=True, dotenv_path="../.env")

True

# pdf -> Markdown 변환
- Docling: 테이블 보존과 RAG 전처리 설명에 적합
  - uv add docling
- PyMuPDF4LLM: 코드가 가장 단순해 입문 실습에 좋음
- Marker: 고급 실습에서 LLM 보정, 복잡한 표 처리까지 확장하기 좋음

In [2]:
# PyMuPDF4LLM 추가 설치 필요 패키지
# uv add python-dotenv langchain-pymupdf4llm chromadb langchain-community langchain-text-splitters langchain-openai langchain-chroma

## pdf를 markdown 변환

In [3]:
from pathlib import Path
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

PDF_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).pdf")
MD_PATH = PDF_PATH.with_suffix(".md")

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF 파일을 찾을 수 없습니다: {PDF_PATH}")

loader = PyMuPDF4LLMLoader(
    str(PDF_PATH),
    mode="page",
    table_strategy="lines",
)

docs = loader.load()

md_text = "\n\n---\n\n".join(
    doc.page_content.strip()
    for doc in docs
    if doc.page_content.strip()
)

MD_PATH.write_text(md_text, encoding="utf-8")

print(f"Markdown 파일 저장 완료: {MD_PATH}")
print(f"추출된 페이지 수: {len(docs)}")
print(f"Markdown 미리보기:\n{md_text[:500]}")

Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Markdown 파일 저장 완료: data\AI@Data_Report_2026_전망_251223(최종).md
추출된 페이지 수: 22
Markdown 미리보기:
|Col1|AI@Data Report<br>토픽 분석을 통한<br>AI 주요 트렌드 및 2026 전망<br>2025 제3호<br>-|Col3|
|---|---|---|
||**목차**<br>❶ **배경 및 목적 /p.06**<br>❷ **자료 분석 및 방법/p.09**<br>❸ **분석 결과/p.12**<br>❹ **2026년 전망: AI 생태계 방향성/p.17**<br>❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|**목차**<br>❶ **배경 및 목적 /p.06**<br>❷ **자료 분석 및 방법/p.09**<br>❸ **분석 결과/p.12**<br>❹ **2026년 전망: AI 생태계 방향성/p.17**<br>❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|
||||

---

### AI@Data Report

**2025 제 3 호**


**(2025. 12월)**

##### 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망


**AI@Data 


## md 파일 전처리하기

In [4]:
from pathlib import Path
from markdown_pre_processing import clean_markdown

# 전처리 스크립트 파일 존재 여부 확인
PREPROCESS_FILE = Path("markdown_pre_processing.py")
if not PREPROCESS_FILE.exists():
    raise FileNotFoundError(
        "markdown_pre_processing.py 파일을 찾을 수 없습니다. "
        "이 파일을 현재 노트북과 같은 폴더에 두세요."
    )
# markdown 전처리
RAW_MD_PATH = Path("data/AI@Data_Report_2026_전망_251223(최종).md")
CLEAN_MD_PATH = Path("data/AI@Data_Report_CLEANED.md")
clean_markdown(RAW_MD_PATH, CLEAN_MD_PATH)


정제된 Markdown 저장 완료: data\AI@Data_Report_CLEANED.md
|Col1|AI@Data Report
토픽 분석을 통한
AI 주요 트렌드 및 2026 전망
2025 제3호
-|Col3|
|---|---|---|
||**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|

---

### AI@Data Report

**2025 제 3 호**

**(2025. 12월)**

##### 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

**AI@Data 보고서 시리즈는...**

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해**

**미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해**

**한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.**

❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의

‘초거대 AI 데이터 발굴 및 기획’ 사업의 산출물이므로, 본 보고서의 내용을

발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다.

‧
❷ 보고서 내용의 무단전재를 금하며, 가공 인용할 때는 반드시 출처를

「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다.

❸ 본 보고서의 내용은 한국지능정보사회진흥원(NIA)의

공식 견해와 다를 수 있습니다.

---

**작성 한국지능정보사회진흥원** 김시연 수석(siyeon@nia.or.kr, 053-230-4249)

추지혜 책임(chuu@nia.or.kr, 053-230-4247)

전

# ChromaDB로 Vector DB 구성

ChromaDB는 로컬 파일에 영구 저장(persist)이 가능한 오픈소스 벡터 DB입니다.

## md 변환 문서 로딩

In [5]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

CLEAN_MD_PATH = Path("data/AI@Data_Report_CLEANED.md")

if not CLEAN_MD_PATH.exists():
    raise FileNotFoundError(f"정제된 Markdown 파일을 찾을 수 없습니다: {CLEAN_MD_PATH}")

loader = TextLoader(str(CLEAN_MD_PATH), encoding="utf-8")
docs = loader.load()

print(f"로드된 문서 수: {len(docs)}")
print("\n[문서 metadata]")
print(docs[0].metadata)

print("\n[문서 미리보기]")
print(docs[0].page_content[:800])

C:\Users\magpi\AppData\Local\Temp\ipykernel_17772\4159142790.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


로드된 문서 수: 1

[문서 metadata]
{'source': 'data\\AI@Data_Report_CLEANED.md'}

[문서 미리보기]
|Col1|AI@Data Report
토픽 분석을 통한
AI 주요 트렌드 및 2026 전망
2025 제3호
-|Col3|
|---|---|---|
||**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|

---

### AI@Data Report

**2025 제 3 호**

**(2025. 12월)**

##### 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

**AI@Data 보고서 시리즈는...**

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 분석을 통해**

**미래 사회의 트렌드를 주도할 수 있는 정책 아젠다를 제시하기 위해**

**한국지능정보사회진흥원(NIA)에서 기획, 발간하는 보고서입니다.**

❶ 본 보고서는 과학기술정보통신부와 한국지능정보사회진흥원(NIA)의

‘초거대 AI 데이터 발굴 및 기획’ 사업의 산출물이므로, 본 보고서의 내용을

발표할 때는 반드시 과학기술정보통신부 사업의 연구결과임을 밝혀야합니다.

‧
❷ 보고서 내용의 무단전재를 금하며, 가공 인용할 때는 반드시 출처를

「한국지능정보사회진흥원(NIA)」이라고 밝혀주시기 바랍니다.

❸ 본 보


## 청크 분할

In [6]:
# Markdown 문서는 전체 파일이 하나의 Document로 로딩된다.
# 따라서 검색 결과를 추적하기 위해 chunk_id를 metadata에 추가한다.

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = splitter.split_documents(docs)

for i, chunk in enumerate(chunks, start=1):
    chunk.metadata["chunk_id"] = i
    chunk.metadata["doc_type"] = "markdown"
    chunk.metadata["source_name"] = CLEAN_MD_PATH.name

print(f"생성된 chunk 수: {len(chunks)}")

print("\n[첫 번째 chunk metadata]")
print(chunks[0].metadata)

print("\n[첫 번째 chunk 내용]")
print(chunks[0].page_content[:500])

생성된 chunk 수: 27

[첫 번째 chunk metadata]
{'source': 'data\\AI@Data_Report_CLEANED.md', 'chunk_id': 1, 'doc_type': 'markdown', 'source_name': 'AI@Data_Report_CLEANED.md'}

[첫 번째 chunk 내용]
|Col1|AI@Data Report
토픽 분석을 통한
AI 주요 트렌드 및 2026 전망
2025 제3호
-|Col3|
|---|---|---|
||**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|**목차**
❶ **배경 및 목적 /p.06**
❷ **자료 분석 및 방법/p.09**
❸ **분석 결과/p.12**
❹ **2026년 전망: AI 생태계 방향성/p.17**
❺ **분석 결과로 본 우리의 대응 및 노력/p.20**|

---

### AI@Data Report

**2025 제 3 호**

**(2025. 12월)**

##### 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망

**AI@Data 보고서 시리즈는...**

**AI와 Data 관련된 국내외 기술 및 정책 동향과 이슈 


## ChromaDB 생성 또는 로딩

In [9]:
from pathlib import Path

from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

# Embedding
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

DB_PATH = Path("chroma_db/ai_trend_report_md")
COLLECTION_NAME = "ai_trend_report_2026_md"

# DB가 이미 존재하면 기존 DB 사용
if DB_PATH.exists():
    print("기존 ChromaDB가 존재하므로 새로 생성하지 않습니다.")
    
    vector_store = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(DB_PATH)
    )

# DB가 없으면 새로 생성
else:
    print("기존 ChromaDB가 없어 새로 생성합니다.")

    vector_store = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=str(DB_PATH)
    )

print(f"저장된 청크 수: {vector_store._collection.count()}")

기존 ChromaDB가 존재하므로 새로 생성하지 않습니다.
저장된 청크 수: 27


# Markdown 기반 RAG 검색 품질 확인

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# top-k 검색을 위한 retriever 생성
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 검색된 문서들을 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages([
    ("system", "아래 문서를 참고해서 질문에 답하세요.\n\n[문서]\n{context}"),
    ("human", "{question}")
])

# LCEL 체인 정의
naive_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [11]:
### 표현 방식에 따른 검색 편차
# [질문]
# 1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
# 2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
# 3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
# 4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
# 5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?


In [12]:
# 두 VectorDB에서 동일 질문으로 검색 비교
question = "2026년 AI 기술 전망에서 추론형 AI는 어떻게 발전하나요?"

# PDF 기반 검색
pdf_results = retriever.invoke(question)
print("[PDF 기반 검색 결과]")
for doc in pdf_results[:2]:
    print(doc.page_content[:200])
    print("---")

# Markdown 기반 검색
md_results = retriever.invoke(question)
print("\n[Markdown 기반 검색 결과]")
for doc in md_results[:2]:
    print(doc.page_content[:200])
    print("---")


[PDF 기반 검색 결과]
**■ 2025년 AI 산업·기술·정책 주요 담론**

(도입 확산과 가치사슬 변화) 기업과 산업 영역에서 AI의 적용 범위가 넓어지며, 시장의 성장

속도·도입 깊이·운영 체계 변화가 주요 쟁점으로 부상함

(지능 구조 고도화와 적용 환경 다변화) 멀티모달·추론·경량화 등 내부 알고리즘이 고도화

되고, 기술 적용이 클라우드에서 온디바이스 등으로 확대되는
---
전문가·산업계 의견을 주기적으로 수렴하고, 이를 관계부처 정책 검토로 연계할 필요가 있음

글로벌 AI 규제 환경과 국내 법·제도 간의 정합성을 지속적으로 점검하여, AI 혁신을 저해하지

않으면서도 저작권 보호와 규제 안정성을 함께 확보할 수 있는 정책 논의 필요

---

#### **토픽 분석을 통한** **AI 주요 트렌드 및 2026 전망**

#
---

[Markdown 기반 검색 결과]
**■ 2025년 AI 산업·기술·정책 주요 담론**

(도입 확산과 가치사슬 변화) 기업과 산업 영역에서 AI의 적용 범위가 넓어지며, 시장의 성장

속도·도입 깊이·운영 체계 변화가 주요 쟁점으로 부상함

(지능 구조 고도화와 적용 환경 다변화) 멀티모달·추론·경량화 등 내부 알고리즘이 고도화

되고, 기술 적용이 클라우드에서 온디바이스 등으로 확대되는
---
전문가·산업계 의견을 주기적으로 수렴하고, 이를 관계부처 정책 검토로 연계할 필요가 있음

글로벌 AI 규제 환경과 국내 법·제도 간의 정합성을 지속적으로 점검하여, AI 혁신을 저해하지

않으면서도 저작권 보호와 규제 안정성을 함께 확보할 수 있는 정책 논의 필요

---

#### **토픽 분석을 통한** **AI 주요 트렌드 및 2026 전망**

#
---


In [ ]:
def inspect_retrieval(query: str, k: int = 5, active_store=vector_store):
    """
    질문에 대해 검색된 문서를 확인한다.

    ChromaDB의 similarity_search_with_score()에서 반환되는 score는 거리값에 가깝다.
    일반적으로 낮을수록 질문과 문서가 더 가깝다.
    """

    # 검색 결과와 score를 함께 반환
    results = active_store.similarity_search_with_score(query, k=k)

    print("=" * 100)
    print(f"[질문] {query}")
    print(f"[검색 개수 k] {k}")
    print("ChromaDB distance score: 낮을수록 더 유사함")
    print("=" * 100)

    for i, (doc, score) in enumerate(results, start=1):
        source = doc.metadata.get("source", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        doc_type = doc.metadata.get("doc_type", "?")

        print(f"\n[{i}] distance={score:.4f} | chunk_id={chunk_id} | doc_type={doc_type}")
        print(f"source={source}")
        print(doc.page_content[:600].replace("\n", " "))

## 표현 방식에 따른 검색 편차

다음 질문들은 의미적으로 유사하지만 표현 방식이 다르다.  
RAG 검색은 질문 문장 자체를 임베딩하므로, 같은 의도라도 표현이 달라지면 검색 결과가 달라질 수 있다.


확인 질문:

1. 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
2. 내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?
3. 합성데이터, 추론형 AI, 멀티모달 기술은 앞으로 어떤 역할을 하나요?
4. AI 정책 분야에서 안전성과 책임이 왜 중요한 이슈인가요?
5. 신뢰할 수 있는 AI를 만들기 위해 어떤 제도적 장치가 필요하다고 하나요?

In [17]:
# 문제 1: 표현 방식에 따른 검색 편차
q1 = "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?"
q2 = "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?"

inspect_retrieval(q1, k=3)

inspect_retrieval(q2, k=3)

[질문] 2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?
[검색 개수 k] 3
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.8247 | chunk_id=4 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
**■ 2025년 AI 산업·기술·정책 주요 담론**  (도입 확산과 가치사슬 변화) 기업과 산업 영역에서 AI의 적용 범위가 넓어지며, 시장의 성장  속도·도입 깊이·운영 체계 변화가 주요 쟁점으로 부상함  (지능 구조 고도화와 적용 환경 다변화) 멀티모달·추론·경량화 등 내부 알고리즘이 고도화  되고, 기술 적용이 클라우드에서 온디바이스 등으로 확대되는 양상이 나타남  (안전·규범 중심의 제도화 심화) 고위험군 평가, 책임 기준 명료화, 데이터·저작권 쟁점 정비  등 위험 관리 중심의 제도 설계가 빠르게 진전되고, 국제 규제와의 정합성을 맞추기 위한  대응도 확대되는 흐름이 확인됨  **■ 2026년 AI 생태계 방향성 전망**  산업·기술·정책이 서로 다른 변화 축을 보이지만, 2026년에는 세 변화가 동시에 심화되며 AI  생태계의 구조 재편이 가속화될 가능성이 있음  - 산업은 도입 확대, 기술은 기능 고도화, 정책은 책임·안전 중심의 제도화가 각각 진전될 것으로  예상되며, 이러한 흐름은 AI 활용의 범위, 속도, 규율 체계에 모두 영향을 미칠 것으로 전망됨  ---  2024년 생성형 AI 확산은 실험적 활용을 넘어 2025년 기업 운영 전반으로

[2] distance=0.8415 | chunk_id=7 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
2025년 AI 생태계에서 부상하는 핵심 이슈와 구조적 변화 를 입체적으로 파악하고자 함  - 생성형 AI 고도화, AI 인프라 경쟁, 안전·책임성 강화, 국제 기준 정비 등 주요 변화 요인을  식별하여 산업·기술·정책 분야의 변화 방

## 관련성이 낮은 청크도 함께 반환되는 문제

기본 검색 방식은 `k=5`로 설정하면 관련성이 낮은 문서가 포함되더라도 최대 5개의 청크를 반환한다.  
이 경우 LLM 프롬프트에 불필요한 정보가 함께 들어가 답변 품질이 낮아질 수 있다.

In [ ]:
### 관련 없는 청크도 반환
# [질문]
# 1. AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
# 2. 보고서에서 말하는 온디바이스 AI 확산은 어떤 의미인가요?
# 3. 토픽모델링에서 단순 출현 빈도보다 중요하게 본 기준은 무엇인가요?
# 4. AI 정책에서 출력물 표시와 데이터 출처 공개는 왜 중요하게 다루어지나요?
# 5. 보고서의 전처리 과정에서 제거한 단어 유형은 무엇인가요?

### 크로마 db의 유사도
- 유사도 계산후 반환 되는 값이 거리값, 값이 작을 수록 유사함.
- 유클리드 거리 제곱 = Squared L2 Distance, 벡터 간 거리의 제곱. 기본값
- cosine 기반 거리를 사용하려면 명시적으로 지정해야함. 

In [19]:
# 문제 2: 관련 없는 청크도 반환
query =  "AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?"

print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] AI 에이전트가 기업 운영 방식에 어떤 변화를 가져올 것으로 전망되나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.9981 | chunk_id=21 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
기업 내부에서 AI 에이전트를 활용한 문서 처리, 고객 지원, 운영 자동화 등이 증가하며  – – 사람 에이전트 시스템이 혼합된 업무 구조 가 일부 영역에서 확산될 가능성이 있음  - 기업 간(B2B) 시스템 연동을 통해 제한된 범위에서 AI 간 협업 형태의 자동화 사례도 등장할  것으로 예상되지만, 그 영향은 아직 초기 단계에 머물며 중장기적 구조 변화의 단초를 형성하는  수준으로 평가됨  ---  ※ IoT 센서와 AI 기술을 통해 물류 및 공급망 전 과정을 자동화하고 최적화 [4)]  **[ 그림 5 ]** **주요 산업 분야별 AI 적용 사례**  2026년 AI 기술은 지능 구조 고도화 와 데이터 한계 보완을 중심 으로 진화할 전망임  - 합성데이터·추론형 AI·멀티모달 기술이 주요 경쟁 축으로 자리 잡으면서, 학습 효율 향상·복합  정보 처리·설명 가능성 강화 등 모델 내부 구조의 질적 개선 흐름이 이어질 것으로 예상됨  기술 고도화는 입력 유형과 데이터 범위를 확장함으로써 AI의 활용 가능성을 넓히고,  복잡한 상황에서의 판단 능력을 강화하는 방향 으로 진화할 것으로 전망됨  - 고품질 데이터 생성·멀티모달·고급 추론 기술이 결합되며 AI의 상황

[2] distance=1.0145 | chunk_id=20 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
구조적 흐름을 도출하였음  - 분석 결과를 통해 세 분야는 서로 다른 영역에서 출발하더라도 활용 확대 → 기술적 고도화  요구 증가 → 제도적 대응 심화 로 이어지는 연속적 흐

## 표·절차 정보 검색 확인

Markdown 변환은 PDF 직접 로딩보다 표와 제목 구조를 더 잘 보존할 수 있다.  
그러나 표가 길거나 청크 경계가 잘못 나뉘면 행·열 관계가 여전히 깨질 수 있다.

확인 질문:

1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [ ]:
## 표·절차 정보 검색 확인
# [질문]
# 1. 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
# 2. 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
# 3. 토픽모델링 전에 제거한 전처리 용어 유형과 예시를 정리해줘.
# 4. AI 산업, AI 기술, AI 정책의 토픽명과 핵심 이슈를 비교해줘.
# 5. 추론형 데이터는 어떤 유형으로 구분되며, 각 유형의 예시는 무엇인가요?

In [ ]:
query =  "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?"

print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] 이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.8266 | chunk_id=9 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
결과로 채택하여 서술함  Ÿ 이는 두 토픽 중 하나가 상대적으로 주제 명확성·키워드 집중도·현행 이슈와의 정합성 측면에서  상대적으로 우세했기 때문  기사·보고서 기반 분석은 실제 AI 트렌드의 전체 맥락과 세부 이슈를 완전히 포괄하기  어려우므로, 향후 연구 및 정책 제안에서는 보다 다양한 데이터 소스 확충과 다층적 분석,  심층 사례 검토가 필요함  Ÿ 언론의 이슈 편향, 국가별 리스크 인식 차이, 기업 홍보성 보도 등으로 인해 담론 기반 분석은  구조적 편향(Structural Bias)이 발생할 여지가 있음  Ÿ 동일 키워드라도 국가별·산업별 맥락이 매우 상이할 수 있어 일반화에는 주의가 필요함  분석 결과는 현재 시점의 데이터에 기반하므로, 향후 기술 발전·정책 변화·시장 흐름에  따라 주요 이슈가 재편될 수 있으며, 지속적 모니터링이 필수적임  Ÿ AI 규제체계, 에이전틱 AI, 글로벌 인프라 경쟁 등은 변화 속도가 빨라, 정기적인 재분석 및  업데이트가 요구됨  ---  9 토픽 분석을 통한 AI 주요 트렌드 및 2026 전망  각 분야별 담론 구조 도출을 위해 본 연구는 데이터 기반 접근법을 적용하였으며, 그 분석  과정은 다음의 단계별 프로세

[2] distance=0.8765 | chunk_id=26 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
source_name=AI@Data_Report_CLEANED.md
데이터에서 벗어나, ‘단계별 과정·의사결정 논

In [18]:
query =  "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
print("유사도 거리 점수 (낮을수록 유사):")
inspect_retrieval(query, k=5)

유사도 거리 점수 (낮을수록 유사):
[질문] 산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?
[검색 개수 k] 5
ChromaDB distance score: 낮을수록 더 유사함

[1] distance=0.7403 | chunk_id=10 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
관련 기사·리포트·정책자료 등 텍스트 데이터를 균형적이고 다원적으로 수집함  - 산업·기술·정책 분야 모두에서 상위 단어가 6~8개 주제 축으로 수렴하는 패턴을 보여 7개  키워드를 채택해 포괄성과 해석성을 확보함  ---  - 분야 간 키워드 수를 동일하게 맞춤으로써 비교 과정에서 기준 차이에 따른 왜곡을 최소화함  본 분석의 분야별 핵심 키워드는 국내외 주요 매체에서 반복 등장한 상위 빈도 핵심어 중  산업·기술·정책의 주요 이슈와 변화 방향을 가장 정확히 대표하는 단어들을 선별하여 구성  - 핵심 키워드는 산업·기술·정책 각 분야에서 반복적으로 등장한 상위 빈도 핵심어 중 대표성이  가장 높은 단어를 기반으로 선정함  - 이는 각 키워드가 분야별 담론 구조와 주요 이슈·변화 방향(시장 확산, 기술 고도화, 규제 강화  등)을 가장 명확하게 설명하는 지표 역할을 하기 때문임  **< 표 2 >** **분야별 데이터 수집 키워드**  |분야|사용 키워드| |---|---| |AI산업|AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환| |AI기술|AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발| |AI정책|

[2] distance=0.8393 | chunk_id=11 | doc_type=markdown
source=data\AI@Data_Report_CLEANED.md
분야별로 설정한 7개 키워드를 기반으로 전체 수집 건수는 282건(분야별 94건)이며, 각  기사·리포트별로 주요 내용 요약본을 정리한 후, 산업·기술·정책별 세부 키워드 분석을 실시함 

# 실행과 답변 비교

## 검색 결과와 최종 답변 확인

In [ ]:
def ask_markdown_rag(question: str):
    """
    Markdown 기반 RAG 체인을 실행하고 답변을 출력한다.
    """

    answer = naive_chain.invoke(question)

    print("=" * 100)
    print(f"[질문]\n{question}")
    print("\n[Markdown 기반 RAG 답변]")
    print(answer)
    print("=" * 100)

    return answer

In [ ]:
rag_questions = [
    "2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?",
    "내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?",
]

for question in rag_questions:
    ask_markdown_rag(question)

[질문]
2026년 AI 기술 전망에서 핵심 변화는 무엇인가요?

[Markdown 기반 RAG 답변]
2026년 AI 기술 전망에서 핵심 변화는 다음과 같습니다:

1. **기능 고도화**: AI 기술의 내부 알고리즘이 멀티모달, 추론, 경량화 등으로 고도화되며, 성능이 향상되고 개발 및 업데이트 주기가 가속화될 것으로 예상됩니다.

2. **적용 범위 확장**: 기술 적용 환경이 클라우드에서 온디바이스로 확대되며, 다양한 디바이스와 서비스에서 AI 기술이 활용될 가능성이 높아집니다.

3. **기술 발전 흐름**: 지능 구조의 고도화 중심으로 기술 발전이 이루어지며, AI의 활용이 더욱 다양화되고 실질적인 사례가 증가할 것으로 보입니다.

이러한 변화들은 AI 기술이 산업 전반에 걸쳐 더욱 깊이 통합되고, 다양한 분야에서의 활용이 증가하는 데 기여할 것입니다.
[질문]
내년 AI 모델은 어떤 방향으로 발전할 것으로 보이나요?

[Markdown 기반 RAG 답변]
2026년 AI 모델은 지능 구조 고도화와 데이터 한계 보완을 중심으로 진화할 것으로 예상됩니다. 주요 발전 방향은 다음과 같습니다:

1. **합성데이터, 추론형 AI, 멀티모달 기술의 발전**: 이러한 기술들이 주요 경쟁 축으로 자리 잡으면서 학습 효율이 향상되고 복합 정보 처리 및 설명 가능성이 강화될 것입니다.

2. **모델 내부 구조의 질적 개선**: 입력 유형과 데이터 범위를 확장함으로써 AI의 활용 가능성이 넓어지고, 복잡한 상황에서의 판단 능력이 강화될 것으로 보입니다.

3. **고품질 데이터의 중요성 증가**: AI 활용이 확산됨에 따라 고품질 및 도메인 특화 데이터의 중요성이 커지며, 데이터 확보, 관리, 유통을 둘러싼 생태계가 빠르게 성장할 것입니다.

4. **AI의 상황 이해 및 문제 해결 능력 향상**: 고급 추론 기술과 멀티모달 기술의 결합으로 AI의 상황 이해와 문제 해결 능력이 향상되며, 서비스 및 산업 전반에서의 활용도가 확대될 전망입니다.

이러한 방향성은 A

In [ ]:
rag_questions = [
    "이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?",
    "산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?"
]

for question in rag_questions:
    ask_markdown_rag(question)

[질문]
이 보고서는 AI 트렌드를 분석하기 위해 어떤 분석 절차를 거쳤나요?

[Markdown 기반 RAG 답변]
이 보고서는 AI 트렌드를 분석하기 위해 다음과 같은 단계별 분석 절차를 거쳤습니다:

1. **데이터 수집**: 2025년 1월부터 11월까지 매주 산업, 기술, 정책 세 분야별로 국내 매체 1건과 해외 매체 1건씩 총 6건을 선정하여 정기적으로 기사 및 보고서를 스크랩하여 분석에 필요한 데이터셋을 누적했습니다.

2. **구조화된 추론형 데이터셋 구축**: 데이터에서 벗어나 단계별 과정, 의사결정 논리, 맥락 등을 포함한 구조화된 추론형 데이터셋을 구축하는 과정을 진행했습니다.

이러한 절차를 통해 AI 트렌드에 대한 보다 체계적이고 심층적인 분석을 수행할 수 있었습니다.
[질문]
산업·기술·정책 분야별 데이터 수집 키워드는 각각 무엇인가요?

[Markdown 기반 RAG 답변]
산업·기술·정책 분야별 데이터 수집 키워드는 다음과 같습니다:

- **AI산업**: AI투자, 시장규모, 기업도입, 생성형AI, 펀딩, 스타트업, 산업전환
- **AI기술**: AI에이전트, 멀티모달, LLM, 온디바이스AI, 오픈소스, 딥시크, 모델개발
- **AI정책**: AI규제, AI기본법, EU AI Act, 거버넌스, 행정명령, 데이터보호, 인프라정책


# 정리

이번 실습에서는 PDF 문서를 Markdown으로 변환한 뒤 RAG를 구성했다.  
PDF 직접 로딩 방식보다 Markdown 변환 방식은 제목, 문단, 표 구조를 더 잘 보존할 수 있다.

### 확인한 내용

1. `pymupdf4llm`을 사용하면 PDF를 Markdown 형식으로 변환할 수 있다.
2. 변환된 Markdown에는 이미지 생략 문구, 반복 제목, 깨진 줄바꿈 같은 노이즈가 포함될 수 있다.
3. Markdown 정제를 통해 불필요한 문구와 반복 헤더를 줄일 수 있다.
4. Markdown 파일은 `TextLoader`로 로딩할 수 있다.
5. `TextLoader`는 기본적으로 페이지 번호 metadata를 제공하지 않는다.
6. Markdown 기반 청크에는 `chunk_id`를 추가해 검색 결과를 추적하는 것이 좋다.
7. 표가 포함된 문서는 chunk 크기가 너무 작으면 행·열 관계가 끊길 수 있다.
8. Markdown 변환은 검색 품질을 개선할 수 있지만, 질문 표현 차이와 관련성이 낮은 청크 반환 문제는 여전히 남는다.